# backward-fn-signature — worked example 1: Write sqrt_back with the canonical (grad_out, out, x) signature

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `backward-fn-signature`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

Every ARENA backward fn shares the shape `(grad_out, out, *args) -> grad_in`. For `out = sqrt(x)` the elementwise derivative is `d(out)/dx = 1/(2*sqrt(x)) = 1/(2*out)`. By the chain rule `dL/dx = grad_out / (2*out)`. This is a case where the cached `out` is the *natural* thing to reuse instead of recomputing the square root.

## Worked solution

**Step 1 — recall the forward.** The forward op is `out = sqrt(x)`. We want `dL/dx` given the upstream gradient `grad_out = dL/dout`.

**Step 2 — local derivative.** `d(sqrt(x))/dx = (1/2) * x**(-1/2) = 1/(2*sqrt(x))`. Since `out = sqrt(x)` is already cached, `1/(2*sqrt(x)) = 1/(2*out)`. Reusing `out` avoids a redundant `sqrt` call — this is exactly why the signature passes `out` to every backward fn.

**Step 3 — chain rule.** Multiply the local derivative by the upstream gradient: `dL/dx = grad_out * 1/(2*out) = grad_out / (2*out)`.

**Step 4 — honor the signature & shapes.** The function takes `(grad_out, out, x)` even though `x` goes unused — uniformity is what lets a dispatcher call every backward fn the same way. The returned tensor is elementwise and therefore has the same shape and dtype as `x`.

In [ ]:
def sqrt_back(grad_out, out, x):
    # out = sqrt(x); d(out)/dx = 1/(2*sqrt(x)) = 1/(2*out).
    # Reuse cached `out` rather than recomputing sqrt. `x` unused.
    return grad_out / (2 * out)

t.manual_seed(0)
x = t.rand(4) + 0.5           # keep strictly positive
out = t.sqrt(x)
grad_out = t.ones_like(out)
grad_x = sqrt_back(grad_out, out, x)
print(grad_x)
print('matches 1/(2 sqrt x):', t.allclose(grad_x, 1 / (2 * t.sqrt(x))))